In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import (Input,Dense,Dropout,BatchNormalization,ReLU,Add,GlobalAveragePooling1D,Concatenate,Conv2D, Reshape)
from sklearn.model_selection import(StratifiedGroupKFold,LeaveOneGroupOut)
from sklearn.metrics import accuracy_score

In [ ]:
#Prepare Data
#Code and Data will be given soon

In [ ]:
seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

In [ ]:
# GCN INPUT
X_gcn = handcrafted_features.astype(np.float32)
# RAW EEG INPUT
X_raw = eeg_data.astype(np.float32)
# ADJACENCY MATRICES
A = adjacency_matrices.astype(np.float32)
# LABELS
y = labels.astype(np.int32)
print("GCN Input Shape:", X_gcn.shape)
print("Raw EEG Shape:", X_raw.shape)
print("Adjacency Shape:", A.shape)
print("Labels Shape:", y.shape)

In [ ]:

class GraphConvolution(tf.keras.layers.Layer):

    def __init__(self, output_dim, order=3, **kwargs):
        super(GraphConvolution, self).__init__(**kwargs)

        self.output_dim = output_dim
        self.order = order

    def build(self, input_shape):

        feature_dim = input_shape[0][-1]

        self.kernel = self.add_weight(
            shape=(feature_dim, self.output_dim, self.order),
            initializer='glorot_uniform',
            trainable=True,
            name='kernel'
        )

        super(GraphConvolution, self).build(input_shape)

    def call(self, inputs):

        X, A = inputs

        out = 0

        # Identity matrix
        A_power = tf.eye(
            tf.shape(A)[-1],
            batch_shape=[tf.shape(A)[0]]
        )

        for k in range(self.order):

            if k > 0:
                A_power = tf.matmul(A_power, A)

            X_k = tf.matmul(A_power, X)

            W_k = self.kernel[:, :, k]

            out += tf.matmul(X_k, W_k)

        return out

In [ ]:
def build_hybrid_model(
    num_nodes=64,
    feature_dim=40,
    time_steps=656,
    num_classes=3,
    fs=160,
    dropout_rate=0.3
):

    X_input = Input(
        shape=(num_nodes, feature_dim),
        name='X_input'
    )

    A_input = Input(
        shape=(num_nodes, num_nodes),
        name='A_input'
    )



    gc1 = GraphConvolution(
        output_dim=4,
        order=3,
        name='GC1'
    )([X_input, A_input])

    gc1 = BatchNormalization()(gc1)
    gc1 = ReLU()(gc1)



    gc2 = GraphConvolution(
        output_dim=8,
        order=3,
        name='GC2'
    )([gc1, A_input])

    gc2 = BatchNormalization()(gc2)
    gc2 = ReLU()(gc2)



    skip1 = Dense(8)(X_input)

    res1 = Add(name='Skip_Add_1')([
        gc2,
        skip1
    ])



    gc3 = GraphConvolution(
        output_dim=16,
        order=3,
        name='GC3'
    )([res1, A_input])

    gc3 = BatchNormalization()(gc3)
    gc3 = ReLU()(gc3)

    gc4 = GraphConvolution(
        output_dim=32,
        order=3,
        name='GC4'
    )([gc3, A_input])

    gc4 = BatchNormalization()(gc4)
    gc4 = ReLU()(gc4)


    skip2 = Dense(32)(gc2)

    res2 = Add(name='Skip_Add_2')([
        gc4,
        skip2
    ])


    gcn_out = GlobalAveragePooling1D(
        name='GAP1'
    )(res2)
#CNN

    raw_input = Input(
        shape=(time_steps, num_nodes),
        name='Raw_EEG_Input'
    )

    x = Reshape(
        (time_steps, num_nodes, 1)
    )(raw_input)


    temporal = Conv2D(
        filters=5,
        kernel_size=(1, fs // 8),
        padding='same',
        activation='relu',
        name='Temporal_Conv'
    )(x)


    spatial = Conv2D(
        filters=5,
        kernel_size=(num_nodes, 1),
        padding='same',
        activation='relu',
        name='Spatial_Conv'
    )(x)



    fusion = Concatenate(axis=-1)([
        temporal,
        spatial
    ])


    feat = Conv2D(
        filters=8,
        kernel_size=(1, 5),
        padding='same',
        activation='relu',
        name='Feature_Conv'
    )(fusion)



    feat = tf.reduce_mean(
        feat,
        axis=2
    )

       # GAP2
    cnn_out = GlobalAveragePooling1D(
        name='GAP2'
    )(feat)


    # FUSION
   =

    merged = Concatenate(
        name='Fusion'
    )([
        gcn_out,
        cnn_out
    ])


    x = Dense(
        16,
        activation='relu'
    )(merged)

    x = Dropout(
        dropout_rate
    )(x)

    output = Dense(
        num_classes,
        activation='softmax'
    )(x)

    model = Model(
        inputs=[
            X_input,
            A_input,
            raw_input
        ],
        outputs=output
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.0005
        ),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model